# VoiceForge — Talking-head VIDEO backend

Turns one photo + your generated narration into a lip-synced MP4, using **SadTalker** (lip sync + blinking + head motion, best for short clips) and **Wav2Lip** (mouth-only, fast enough for long narration). Exposes a public HTTPS API via a Cloudflare tunnel.

**This is a SEPARATE backend from the TTS notebook.** It needs an older torch/numpy stack that conflicts with Chatterbox, so run it in its own Colab session — after your speech is done, so the single free GPU is reused rather than needed twice at once.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Then **Run all**. Cell 1 installs both engines (~8–12 min the first time). Cell 3 prints the backend URL — paste it into the VoiceForge **Video** tab. Keep this tab open while rendering.

In [ ]:
# Cell 1 — Install (SadTalker + Wav2Lip). ~8–12 min the first time.
import os

# Shared stack SadTalker is known-good on. torchvision 0.15.2 still ships
# torchvision.transforms.functional_tensor, which basicsr/gfpgan import —
# newer torchvision removes it and breaks them. Do NOT install SadTalker's
# requirements.txt: it would drag torch back to 1.12 and undo this.
!pip install -q numpy==1.23.5 torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2
!pip install -q fastapi uvicorn nest-asyncio python-multipart soundfile imageio imageio-ffmpeg
!pip install -q facexlib==0.3.0 gfpgan insightface==0.7.3 onnxruntime moviepy \
  opencv-python-headless yacs kornia==0.6.8 pydub librosa==0.10.1 numba resampy \
  scikit-image tqdm safetensors av basicsr==1.4.2 face-alignment==1.3.5

# --- SadTalker + checkpoints
if not os.path.exists('/content/SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
!cd /content/SadTalker && bash scripts/download_models.sh

# --- Wav2Lip (justinjohn0306 fork: newer-torch friendly + hosts checkpoints)
if not os.path.exists('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
!mkdir -p /content/Wav2Lip/checkpoints /content/Wav2Lip/face_detection/detection/sfd
!wget -q 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth' -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth
!wget -q 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/s3fd.pth' -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth

# --- Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print('Installed. Run Cell 2. (Red pip conflict lines about unrelated')
print('packages are safe to ignore — same as the TTS notebook.)')

In [ ]:
# Cell 2 — Verify GPU + detect which engines are ready
import torch, os, glob
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'
GPU = torch.cuda.get_device_name(0)
print(GPU)

# checkpoints present is enough; gfpgan weights auto-download on first run
SAD_OK = bool(glob.glob('/content/SadTalker/checkpoints/*'))
W2L_OK = os.path.exists('/content/Wav2Lip/checkpoints/wav2lip_gan.pth') and \
         os.path.exists('/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth')
print('sadtalker ready:', SAD_OK)
print('wav2lip ready:  ', W2L_OK)
if not (SAD_OK or W2L_OK):
    print('Neither engine set up — re-run Cell 1 and watch for download errors.')

In [ ]:
# Cell 3 — API server + public URL
import io, os, base64, subprocess, threading, time, re, glob, shutil, uuid
import soundfile as sf, torch, nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])

WORK = '/content/vf_video'; os.makedirs(WORK, exist_ok=True)

def _engines():
    e = []
    if SAD_OK: e.append('sadtalker')
    if W2L_OK: e.append('wav2lip')
    return e

def _run(cmd, cwd):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout or 'unknown error')[-1600:])

def run_sadtalker(image_path, audio_path, out_dir):
    # --still reduces head jitter; --preprocess full keeps the whole photo
    # framed (vs 'crop' which zooms to the face); gfpgan sharpens the result.
    _run(['python', 'inference.py', '--driven_audio', audio_path,
          '--source_image', image_path, '--result_dir', out_dir,
          '--still', '--preprocess', 'full', '--enhancer', 'gfpgan'],
         cwd='/content/SadTalker')
    mp4s = sorted(glob.glob(os.path.join(out_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime)
    if not mp4s: raise RuntimeError('SadTalker produced no mp4')
    return mp4s[-1]

def run_wav2lip(image_path, audio_path, out_path):
    _run(['python', 'inference.py', '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
          '--face', image_path, '--audio', audio_path, '--outfile', out_path,
          '--nosmooth', '--pads', '0', '15', '0', '0'],
         cwd='/content/Wav2Lip')
    if not os.path.exists(out_path): raise RuntimeError('Wav2Lip produced no mp4')
    return out_path

@app.get('/health')
def health():
    return {'ok': True, 'gpu': GPU, 'engines': _engines()}

@app.post('/animate')
async def animate(payload: dict):
    engine = payload.get('engine', 'sadtalker')
    if engine not in _engines():
        return JSONResponse({'error': f'{engine} not loaded'}, status_code=400)
    job = os.path.join(WORK, uuid.uuid4().hex[:10]); os.makedirs(job, exist_ok=True)
    try:
        ext = 'jpg' if 'jpeg' in payload.get('image_mime', '') or 'jpg' in payload.get('image_mime', '') else 'png'
        img = os.path.join(job, f'face.{ext}')
        wav = os.path.join(job, 'audio.wav')
        with open(img, 'wb') as f: f.write(base64.b64decode(payload['image_b64']))
        with open(wav, 'wb') as f: f.write(base64.b64decode(payload['audio_b64']))
        info = sf.info(wav); dur = info.frames / info.samplerate
        t0 = time.time()
        if engine == 'sadtalker':
            mp4 = run_sadtalker(img, wav, os.path.join(job, 'out'))
        else:
            mp4 = run_wav2lip(img, wav, os.path.join(job, 'out.mp4'))
        with open(mp4, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        return {'video_b64': b64, 'duration': round(dur, 2),
                'gen_seconds': round(time.time() - t0, 2)}
    except Exception as e:
        return JSONResponse({'error': str(e)[-1600:]}, status_code=500)
    finally:
        shutil.rmtree(job, ignore_errors=True)
        torch.cuda.empty_cache()

# --- start server + tunnel ---
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error'),
    daemon=True,
).start()
time.sleep(3)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        print('\n' + '=' * 60)
        print('  VIDEO BACKEND URL — paste into the VoiceForge Video tab:')
        print('  ' + m.group(0))
        print('=' * 60 + '\n')
        print('Engines available:', _engines())
        print('Keep this tab open while rendering.')
        break

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab disconnects idle tabs. Run this in the background during long renders.
import time
while True:
    time.sleep(60)
    print('.', end='', flush=True)